# 🎬 Hikâye Stüdyosu — Tek Parça Sahne (karakter sahnenin içinde)
Karakteri **sahnenin içinde, tek resim olarak** üretir: ışık, gölge ve perspektif kendiliğinden uyumlu olur.
Uygulamada bu resmi **1. bölümde karakter olarak** seç; uygulama yüzü bulup konuşturur.

**Başlamadan:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*. Hücreleri sırayla ▶ çalıştır.

## 1) Kurulum (~1 dk)

In [ ]:
# Kurulum: Colab'daki hazır paketleri DEĞİŞTİRMEDEN sadece eksikleri ekler
!pip -q install diffusers pooch pymatting onnxruntime
!pip -q install --no-deps rembg
print('Kurulum tamam ✅  Şimdi alttaki hücreye geç.')

## 1b) Görsel modelini yükle (~5-7 dk)
Hata çıkarsa: *Çalışma zamanı → Oturumu yeniden başlat*, sonra sadece bu hücreyi tekrar çalıştır.

In [ ]:
import torch, os, json, random
from PIL import Image, ImageDraw
from google.colab import drive
drive.mount('/content/drive')
ANA = '/content/drive/MyDrive/AnimasyonStudyo'
for k in ['hazir_sahneler']: os.makedirs(f'{ANA}/{k}', exist_ok=True)
assert torch.cuda.is_available(), 'GPU YOK: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seç'
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0',
        vae=vae, torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')
STIL = '3d animated movie style, pixar-like stylized cartoon character, soft subsurface lighting, big expressive eyes, smooth skin, high quality 3d render'
NEG = 'photo, realistic, ugly, deformed, extra fingers, bad anatomy, text, watermark, blurry, scary, open mouth, teeth, cropped head'
print('Hazır ✅')

## 2) Senaryonun sahnesini üret
Alanları doldurup ▶ bas. 4 aday üretilir (~3 dk). Beğenmezsen tekrar çalıştır, her seferinde yeni adaylar gelir.
Hepsi Drive'da `AnimasyonStudyo/hazir_sahneler` klasörüne kaydedilir; beğendiğini oradan indir.

In [ ]:
ad = 'senaryo1' #@param {type:'string'}
yas = 15 #@param {type:'slider', min:15, max:65, step:1}
cinsiyet = 'Kadın' #@param ['Kadın','Erkek']
gorunus_ingilizce = 'long brown hair in a ponytail, wearing a white school shirt and navy tie' #@param {type:'string'}
mekan = 'Yatak odası' #@param ['Yatak odası','Oturma odası','Okul bahçesi','Sınıf','Kütüphane','Hastane odası','Park','Mutfak','Ofis','Balkon']
poz = 'Oturuyor' #@param ['Oturuyor','Masada oturuyor','Ayakta']
saat = 'Gündüz' #@param ['Gündüz','Akşam','Gece']
adet = 4 #@param {type:'slider', min:1, max:4, step:1}

if yas < 18: kim = 'teenage girl' if cinsiyet == 'Kadın' else 'teenage boy'
elif yas >= 58: kim = 'old woman' if cinsiyet == 'Kadın' else 'old man'
else: kim = 'woman' if cinsiyet == 'Kadın' else 'man'
M = {'Yatak odası':'her cozy bedroom' if cinsiyet=='Kadın' else 'his cozy bedroom', 'Oturma odası':'a cozy turkish living room',
     'Okul bahçesi':'a school courtyard with trees', 'Sınıf':'a classroom', 'Kütüphane':'a cozy library with bookshelves',
     'Hastane odası':'a calm hospital room', 'Park':'a green city park', 'Mutfak':'a warm home kitchen',
     'Ofis':'a small office', 'Balkon':'a balcony with plants overlooking the city'}
P = {'Oturuyor':'sitting', 'Masada oturuyor':'sitting at a table with hands resting on it', 'Ayakta':'standing'}
Z = {'Gündüz':'soft daylight', 'Akşam':'warm golden evening light', 'Gece':'night, warm lamp light'}
p = (f'{STIL}, a {yas} year old Turkish {kim}, {gorunus_ingilizce}, {P[poz]} in {M[mekan]}, {Z[saat]}, '
     'medium shot from the waist up, character in the center of the frame, facing the camera, looking at the viewer, '
     'calm gentle expression, mouth closed, cinematic composition, soft depth of field background')
neg = NEG + ', full body, legs, feet, multiple people, two people, looking away, side view, profile'
k = f'{ANA}/hazir_sahneler'
from PIL import Image
sonuclar = []
for i in range(adet):
    s = random.randint(0, 2**31-1)
    img = pipe(p, negative_prompt=neg, num_inference_steps=32, guidance_scale=6.0, width=1344, height=768,
               generator=torch.Generator('cuda').manual_seed(s)).images[0]
    yol = f'{k}/{ad}_{s % 100000}.png'; img.save(yol); sonuclar.append(yol)
    print(f'{i+1}/{adet} kaydedildi: {yol.split("/")[-1]}'); display(img.resize((672, 384)))
print('Bitti ✅ Drive → AnimasyonStudyo → hazir_sahneler')